### Import a libraries

In [2]:
import os, glob, random, json, warnings
import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

warnings.filterwarnings('ignore')

### Define main parameters

In [28]:
CONFIG = {
    "data_dir":     r"E:\car_image_dataset\ltsm_dataset\model_train",
    "output_dir":   "./outputs",
    "obs_len":      20,      # 2 сек спостереження @ 10 Hz
    "pred_len":     30,      # 3 сек прогноз      @ 10 Hz
    "hidden_size":  128,
    "num_layers":   2,
    "dropout":      0.3,
    "batch_size":   64,
    "lr":           1e-3,
    "epochs":       20,
    "val_split":    0.15,
    "test_split":   0.10,
    "seed":         42,
    "device":       "cuda" if torch.cuda.is_available() else "cpu",
    "max_samples":  10000,
    "cache_dir":     r"E:\car_image_dataset\ltsm_dataset\cache"
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
os.makedirs(CONFIG["output_dir"], exist_ok=True)

In [33]:
import pickle


def load_argoverse(data_dir, obs_len, pred_len, max_samples=None):

    os.makedirs(CONFIG["cache_dir"], exist_ok=True)

    cache_name = f"argoverse_obs{obs_len}_pred{pred_len}_max{max_samples}.json"
    cach_path = os.path.join(CONFIG["cache_dir"], cache_name)

    if os.path.exists(cach_path):
        print("Завантаження даних з кешу...")
        with open(cach_path, "rb") as f:
            return pickle.load(f)

    for f in os.listdir(CONFIG["cache_dir"]):
        if f.endswith(".pkl") or f.endswith(".json"):
            os.remove(os.path.join(CONFIG["cache_dir"], f))
    print("Старий кеш видалено!!!")

    csvs = glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True)

    if not csvs:
        return None
    print(f"✔ Знайдено {len(csvs)} CSV файлів")
    
    seqs = []
    for f in tqdm(csvs[:max_samples], desc="Завантаження"):
        try:
            df = pd.read_csv(f)
            agent = df[df["OBJECT_TYPE"]=="AGENT"].sort_values("TIMESTAMP")
            if len(agent) < obs_len + pred_len:
                continue

            
            # seqs.append({"xs": agent["X"].values[:obs_len+pred_len],
            #              "ys": agent["Y"].values[:obs_len+pred_len],
            #              "maneuver": "real"})

            x0 = agent["X"].values[obs_len - 1]
            y0 = agent["Y"].values[obs_len - 1]

            xs = agent["X"].values[:obs_len + pred_len] - x0
            ys = agent["Y"].values[:obs_len + pred_len] - y0

            # 🔑 2️⃣ obs / future
            obs = np.stack([xs[:obs_len], ys[:obs_len]], axis=1)      # (20, 2)
            tgt = np.stack([xs[obs_len:], ys[obs_len:]], axis=1)     # (30, 2)

            # 🔑 3️⃣ масштабування
            obs /= 10.0
            tgt /= 10.0

            seqs.append((obs.astype("float32"), tgt.astype("float32")))

        except Exception: 
            continue

    with open(cach_path, "wb") as f:
        pickle.dump(seqs, f)
        
    print(f"Дані збережено в кеші: {cach_path}")

    return seqs or None

In [34]:
class ArgoverseDataset(Dataset):
    """Input: (obs_len, 4) = [x, y, vx, vy] | Target: (pred_len, 2) = [x, y]"""
    def __init__(self, seqs, obs_len, pred_len, scaler=None, fit_scaler=False):
        self.obs_len, self.pred_len, self.samples = obs_len, pred_len, []
        all_feats = []

        # seqs — список кортежів (obs_array, tgt_array)
        # obs_array: (obs_len, 2), tgt_array: (pred_len, 2)
        for obs, tgt in seqs:
            xs = obs[:, 0]
            ys = obs[:, 1]
            vx = np.diff(xs, prepend=xs[0]) / 0.1
            vy = np.diff(ys, prepend=ys[0]) / 0.1
            all_feats.append(np.stack([xs, ys, vx, vy], axis=1))  # (obs_len, 4)

        flat = np.concatenate(all_feats, axis=0)
        if fit_scaler:
            scaler = StandardScaler().fit(flat)
        self.scaler = scaler

        for i, (obs, tgt) in enumerate(seqs):
            feats = scaler.transform(all_feats[i]) if scaler else all_feats[i]
            obs_t  = torch.tensor(feats[:obs_len], dtype=torch.float32)
            target = torch.tensor(tgt[:pred_len],  dtype=torch.float32)  # (pred_len, 2)
            self.samples.append((obs_t, target, "unknown"))

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i):
        obs, target, _ = self.samples[i]
        return obs, target


# LSTM

In [22]:
class MotionLSTM(nn.Module):
    """
    Encoder-Decoder LSTM

    Вхід:  (B, obs_len=20, 4)  — x, y, vx, vy
    Вихід: (B, pred_len=30, 2) — x, y

    Encoder → приховані стани → Decoder (авторегресивний) → FC → (x, y)
    """
    def __init__(self, input_size=4, hidden=128, layers=2, pred_len=30, drop=0.3):
        super().__init__()
        self.pred_len = pred_len

        self.encoder = nn.LSTM(input_size, hidden, layers,
                               batch_first=True, dropout=drop if layers>1 else 0.)
        self.decoder = nn.LSTM(2, hidden, layers,
                               batch_first=True, dropout=drop if layers>1 else 0.)
        self.fc = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(drop), nn.Linear(64, 2))

    def forward(self, x, target=None, tf_ratio=0.5):
        _, (h, c) = self.encoder(x)
        dec = x[:, -1, :2].unsqueeze(1)
        outs = []
        for t in range(self.pred_len):
            out, (h, c) = self.decoder(dec, (h, c))
            pred = self.fc(out.squeeze(1))
            outs.append(pred.unsqueeze(1))
            dec = (target[:, t, :].unsqueeze(1)
                   if target is not None and random.random() < tf_ratio
                   else pred.unsqueeze(1))
        return torch.cat(outs, dim=1)

    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ──────── МЕТРИКИ ────────
def ade(p, t): return torch.mean(torch.sqrt(((p-t)**2).sum(-1))).item()
def fde(p, t): return torch.mean(torch.sqrt(((p[:,-1]-t[:,-1])**2).sum(-1))).item()


In [23]:
def train_epoch(model, loader, opt, crit, dev, epoch, total):
    model.train()
    tl, ta, tf_ = 0, 0, 0
    tf_ratio = max(0., 0.5 * (1 - epoch/total))
    for obs, tgt in tqdm(loader, desc=f"Epoch {epoch+1}/{total}", leave=False):
        obs, tgt = obs.to(dev), tgt.to(dev)
        pred = model(obs, tgt, tf_ratio)
        loss = crit(pred, tgt)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tl += loss.item(); ta += ade(pred, tgt); tf_ += fde(pred, tgt)
    n = len(loader)
    return tl/n, ta/n, tf_/n

@torch.no_grad()
def eval_epoch(model, loader, crit, dev):
    model.eval()
    tl, ta, tf_ = 0, 0, 0
    for obs, tgt in loader:
        obs, tgt = obs.to(dev), tgt.to(dev)
        pred = model(obs, tf_ratio=0.)
        tl += crit(pred, tgt).item(); ta += ade(pred, tgt); tf_ += fde(pred, tgt)
    n = len(loader)
    return tl/n, ta/n, tf_/n

In [35]:
def plot_curves(h, path):
    fig = plt.figure(figsize=(16, 10), facecolor='#0d1117')
    gs  = GridSpec(2, 3, hspace=0.45, wspace=0.35)
    tc  = dict(tr='#58a6ff', vl='#f78166', grid='#21262d', txt='#c9d1d9')

    pairs = [("train_loss","val_loss","MSE Loss"),
             ("train_ade","val_ade","ADE (m)"),
             ("train_fde","val_fde","FDE (m)")]

    for col, (tk, vk, title) in enumerate(pairs):
        for row in range(2):
            ax = fig.add_subplot(gs[row, col])
            ax.set_facecolor('#161b22')
            ax.tick_params(colors=tc['txt'], labelsize=8)
            for sp in ax.spines.values(): sp.set_edgecolor(tc['grid'])
            ax.grid(color=tc['grid'], lw=0.6, alpha=0.7)
            ep = range(1, len(h[tk])+1)

            if row == 0:
                ax.plot(ep, h[tk], color=tc['tr'], lw=2, label='Train')
                ax.plot(ep, h[vk], color=tc['vl'], lw=2, ls='--', label='Val')
                bv = min(h[vk]); bi = h[vk].index(bv)+1
                ax.axvline(bi, color='#3fb950', lw=1.2, ls=':', alpha=0.8)
                ax.scatter([bi], [bv], color='#3fb950', s=50, zorder=5)
                ax.legend(fontsize=8, framealpha=0, labelcolor=tc['txt'])
                ax.set_title(title, color=tc['txt'], fontsize=10, pad=6)
            else:
                impr = [(h[vk][0]-v)/max(h[vk][0],1e-9)*100 for v in h[vk]]
                bars = ['#3fb950' if v>=0 else '#f78166' for v in impr]
                ax.bar(ep, impr, color=bars, alpha=0.8, width=0.7)
                ax.axhline(0, color=tc['txt'], lw=0.8, alpha=0.5)
                ax.set_title(f"{title} improvement %", color=tc['txt'], fontsize=9, pad=6)
            ax.set_xlabel('Epoch', color=tc['txt'], fontsize=8)

    fig.suptitle('LSTM Training Dashboard — Argoverse Motion Forecasting',
                 color='white', fontsize=14, fontweight='bold', y=0.98)
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.close()
    print(f"  📊 Графіки збережено: {path}")

In [36]:
def plot_predictions(model, dataset, dev, save_path):
    model.eval()
    cols = dict(straight='#58a6ff', curve_left='#3fb950', curve_right='#f78166',
                lane_change='#d2a8ff', stop_go='#ffa657', real='#79c0ff', unknown='#8b949e')

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), facecolor='#0d1117')
    fig.suptitle('Trajectory Predictions — LSTM vs Ground Truth',
                 color='white', fontsize=14, fontweight='bold')

    idxs = random.sample(range(len(dataset)), 6)
    with torch.no_grad():
        for i, si in enumerate(idxs):
            obs_t, tgt_t = dataset[si]
            pred = model(obs_t.unsqueeze(0).to(dev), tf_ratio=0.).squeeze(0).cpu().numpy()
            tgt  = tgt_t.numpy()
            obs  = obs_t[:, :2].numpy()
            m    = dataset.samples[si][2]

            ax = axes[i//3][i%3]
            ax.set_facecolor('#161b22')
            ax.tick_params(colors='#8b949e', labelsize=7)
            for sp in ax.spines.values(): sp.set_edgecolor('#21262d')
            ax.grid(color='#21262d', lw=0.5, alpha=0.6)

            ax.plot(obs[:,0], obs[:,1], 'o-', color='#8b949e', ms=3, lw=1.5, label='Observed')
            ax.plot(tgt[:,0], tgt[:,1], 's--', color='#3fb950', ms=3, lw=1.5, label='GT Future')
            ax.plot(pred[:,0],pred[:,1], '^-', color='#f78166', ms=3, lw=1.8, label='LSTM Pred')
            ax.scatter(*obs[0],  color='white',   s=50, zorder=6)
            ax.scatter(*tgt[-1], color='#3fb950', s=60, zorder=6, marker='*')
            ax.scatter(*pred[-1],color='#f78166', s=60, zorder=6, marker='*')

            err = np.sqrt(((pred[-1]-tgt[-1])**2).sum())
            ax.set_title(f'{m.upper()} | FDE={err:.2f}m', color=cols.get(m,'#8b949e'), fontsize=9)
            ax.legend(fontsize=7, framealpha=0, labelcolor='#c9d1d9')

    plt.tight_layout(rect=[0,0,1,0.96])
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.close()
    print(f"  🗺  Траєкторії збережено: {save_path}")

In [37]:
def plot_errors(model, loader, dev, path):
    model.eval()
    ade_l, fde_l = [], []
    with torch.no_grad():
        for obs, tgt in loader:
            obs, tgt = obs.to(dev), tgt.to(dev)
            pred = model(obs, tf_ratio=0.)
            ade_l.extend(torch.sqrt(((pred-tgt)**2).sum(-1)).mean(-1).cpu().numpy())
            fde_l.extend(torch.sqrt(((pred[:,-1]-tgt[:,-1])**2).sum(-1)).cpu().numpy())

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor='#0d1117')
    for ax, data, label, clr in [
        (axes[0], ade_l, 'Average Displacement Error (ADE)', '#58a6ff'),
        (axes[1], fde_l, 'Final Displacement Error (FDE)',   '#f78166'),
    ]:
        ax.set_facecolor('#161b22')
        ax.tick_params(colors='#c9d1d9')
        for sp in ax.spines.values(): sp.set_edgecolor('#21262d')
        ax.grid(color='#21262d', lw=0.5, alpha=0.6)
        ax.hist(data, bins=40, color=clr, alpha=0.8, edgecolor='#0d1117', lw=0.3)
        ax.axvline(np.mean(data),   color='white',   lw=1.5, label=f'Mean {np.mean(data):.2f}m')
        ax.axvline(np.median(data), color='#3fb950', lw=1.5, ls='--', label=f'Median {np.median(data):.2f}m')
        ax.set_xlabel('Error (m)', color='#c9d1d9'); ax.set_ylabel('Count', color='#c9d1d9')
        ax.set_title(label, color='white', fontsize=10)
        ax.legend(fontsize=8, framealpha=0, labelcolor='#c9d1d9')

    fig.suptitle('Error Distribution — Test Set', color='white', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.close()
    print(f"  📈 Розподіл помилок збережено: {path}")

In [38]:
def main():
    print("\n" + "="*60)
    print("  Argoverse LSTM Motion Forecasting")
    print("="*60)

    seqs = load_argoverse(CONFIG["data_dir"], CONFIG["obs_len"],
                          CONFIG["pred_len"], CONFIG["max_samples"])
    if seqs is None:
        print("\n❌ Немає даних для тренування")
        return
    print(f"Всього послідовностей: {len(seqs)}")

    ds = ArgoverseDataset(seqs, CONFIG["obs_len"], CONFIG["pred_len"], fit_scaler=True)

    n = len(ds)
    n_test = int(n * CONFIG["test_split"])
    n_val  = int(n * CONFIG["val_split"])
    n_tr   = n - n_val - n_test
    tr_ds, vl_ds, ts_ds = random_split(ds, [n_tr, n_val, n_test],
                                        generator=torch.Generator().manual_seed(CONFIG["seed"]))
    print(f"Train={n_tr} | Val={n_val} | Test={n_test}")

    tr_ld = DataLoader(tr_ds, CONFIG["batch_size"], shuffle=True,  num_workers=0)
    vl_ld = DataLoader(vl_ds, CONFIG["batch_size"], shuffle=False, num_workers=0)
    ts_ld = DataLoader(ts_ds, CONFIG["batch_size"], shuffle=False, num_workers=0)

    model = MotionLSTM(hidden=CONFIG["hidden_size"], layers=CONFIG["num_layers"],
                       pred_len=CONFIG["pred_len"], drop=CONFIG["dropout"]).to(CONFIG["device"])
    print(f"\nПараметрів: {model.n_params():,}\n{model}\n")

    crit  = nn.MSELoss()
    opt   = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

    h = {k: [] for k in ["train_loss","val_loss","train_ade","val_ade","train_fde","val_fde"]}
    best_val, best_path = float('inf'), os.path.join(CONFIG["output_dir"], "best_model.pt")

    for ep in range(CONFIG["epochs"]):
        tr_l, tr_a, tr_f = train_epoch(model, tr_ld, opt, crit, CONFIG["device"], ep, CONFIG["epochs"])
        vl_l, vl_a, vl_f = eval_epoch(model, vl_ld, crit, CONFIG["device"])
        sched.step(vl_l)
        for k, v in zip(h.keys(), [tr_l, vl_l, tr_a, vl_a, tr_f, vl_f]):
            h[k].append(v)

        flag = ""
        if vl_l < best_val:
            best_val = vl_l
            torch.save({"epoch": ep, "model_state": model.state_dict()}, best_path)
            flag = " ← BEST"
        print(f"  Ep {ep+1:2d}/{CONFIG['epochs']} | L {tr_l:.4f}/{vl_l:.4f} | "
              f"ADE {tr_a:.3f}/{vl_a:.3f} | FDE {tr_f:.3f}/{vl_f:.3f}{flag}")

    model.load_state_dict(torch.load(best_path, map_location=CONFIG["device"])["model_state"])
    ts_l, ts_a, ts_f = eval_epoch(model, ts_ld, crit, CONFIG["device"])
    print(f"\n  TEST: Loss={ts_l:.4f} | ADE={ts_a:.3f}m | FDE={ts_f:.3f}m")

    plot_curves(h,      os.path.join(CONFIG["output_dir"], "training_curves.png"))
    plot_predictions(model, ds, CONFIG["device"],
                     os.path.join(CONFIG["output_dir"], "trajectory_predictions.png"))
    plot_errors(model, ts_ld, CONFIG["device"],
                os.path.join(CONFIG["output_dir"], "error_distribution.png"))

    json.dump({"test": {"loss":ts_l,"ade":ts_a,"fde":ts_f}, "history": h},
              open(os.path.join(CONFIG["output_dir"],"results.json"),"w"), indent=2)

    print("\n✅ Готово! Всі файли у ./outputs/")


if __name__ == "__main__":
    main()


  Argoverse LSTM Motion Forecasting
Завантаження даних з кешу...
Всього послідовностей: 10000
Train=7500 | Val=1500 | Test=1000

Параметрів: 408,770
MotionLSTM(
  (encoder): LSTM(4, 128, num_layers=2, batch_first=True, dropout=0.3)
  (decoder): LSTM(2, 128, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)



  Ep  1/20 | L 0.4531/0.2335 | ADE 0.647/0.512 | FDE 1.270/1.014 ← BEST


  Ep  2/20 | L 0.0842/0.1652 | ADE 0.298/0.395 | FDE 0.571/0.850 ← BEST


  Ep  3/20 | L 0.0730/0.1228 | ADE 0.279/0.318 | FDE 0.524/0.816 ← BEST


  Ep  4/20 | L 0.0735/0.1804 | ADE 0.277/0.401 | FDE 0.531/0.960


...

  Ep 16/20 | L 0.1056/0.1252 | ADE 0.325/0.331 | FDE 0.652/0.739


  Ep 17/20 | L 0.1188/0.1275 | ADE 0.344/0.332 | FDE 0.701/0.755


  Ep 18/20 | L 0.1216/0.1054 | ADE 0.347/0.292 | FDE 0.713/0.635 ← BEST


  Ep 19/20 | L 0.1323/0.1041 | ADE 0.359/0.291 | FDE 0.752/0.650 ← BEST


  Ep 20/20 | L 0.1360/0.1005 | ADE 0.363/0.282 | FDE 0.761/0.640 ← BEST

  TEST: Loss=0.0910 | ADE=0.271m | FDE=0.616m
  📊 Графіки збережено: ./outputs\training_curves.png
  🗺  Траєкторії збережено: ./outputs\trajectory_predictions.png
  📈 Розподіл помилок збережено: ./outputs\error_distribution.png

✅ Готово! Всі файли у ./outputs/
